In [1]:
#import data
import retinanalysis as ra
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import date

/opt/miniconda3/envs/retinanalysis/lib/python3.11/site-packages/datajoint/settings.py:992: UserWarning: No datajoint.json found. Using defaults and environment variables. Run `dj.config.save_template()` to create a template configuration.
  config = _create_config()


In [2]:
exp_name = '20250715C'
datafile_name = 'data016'

In [3]:
pipe = ra.create_mea_pipeline(exp_name, datafile_name, typing_file = 'dragos_kilosort2.5.classification.txt')

[2026-06-29 15:51:05] DataJoint 2.2.2 connected to root@127.0.0.1:3306
Initializing StimBlock for 20250715C block 56
For Rig C 20250715C:
This could be due to PatternMode usage.
d_display will keep the min: 59.0
{'disp_type': 'LCR', 'mu_per_pixel': 3.34, 'n_ht': 1140, 'n_wt': 1824, 'mean_frame_rate': 59.94154881781792, 'stage_frame_rate': np.float64(59.0)}
Nearest noise chunk for data016 is chunk3 with distance 127 minutes.

Initializing ResponseBlock for 20250715C block 56
Assuming there are 12 completed epochs.
Loading VCD from /Users/simontesfaye/Documents/data/analysis/20250715C/data016/kilosort2.5 ...
VCD loaded with 786 cells.

Using chunk3 for AnalysisChunk

Loading VCD from /Users/simontesfaye/Documents/data/analysis/20250715C/chunk3/kilosort2.5 ...
VCD loaded with 661 cells.


Loaded spatial maps for channels [0, 2] and 661 cells of shape (127, 203, 2)
Spatial maps have been padded to align with RF parameters.

Cluster matching 20250715C chunk3 with ObjectMotionDots ...
66.87%

## Get experiement date (Reusable)

In [6]:
def get_experiment_date(stim_block):
    exp_name = stim_block.exp_name

    year = int(exp_name[0:4])
    month = int(exp_name[4:6])
    day = int(exp_name[6:8])

    return date(year, month, day)

## Get spike counts (Reusable)

In [7]:
def get_spike_count_table(response_block, cell_types=None, minimum_n=3):
    spike_times = ra.get_spike_xarr(
        response_block,
        cell_types=cell_types,
        minimum_n=minimum_n
    )

    spike_counts = xr.apply_ufunc(
        len,
        spike_times,
        vectorize=True
    )

    spike_counts = spike_counts.to_dataframe(
        name="spike_count"
    ).reset_index()

    return spike_counts

## Add parameter column (Reusable) Aligns epoch with constant values

In [14]:
def add_parameter_column(spike_counts, stim_block, parameter_name):
    values = stim_block.d_epoch_block_params[parameter_name]

    epoch_to_value = dict(enumerate(values))

    spike_counts[parameter_name] = spike_counts["epoch"].map(epoch_to_value)

    return spike_counts

## Average responses by parameter (Reusable)

In [8]:
def average_response_by_parameter(spike_counts, parameter_name):
    summary = (
        spike_counts
        .groupby(["cell_id", parameter_name])["spike_count"]
        .mean()
        .reset_index()
    )

    return summary

## Metric index 

In [9]:
def normalized_difference(low_response, high_response):
    return (high_response - low_response) / (
        high_response + low_response
    )

## Apply metric to each cell

In [10]:
def calculate_metric_table(
    summary,
    parameter_name,
    metric_func,
    metric_name
):
    rows = []

    for cell_id in summary["cell_id"].unique():
        cell_data = summary[summary["cell_id"] == cell_id]

        smallest_value = cell_data[parameter_name].min()
        largest_value = cell_data[parameter_name].max()

        low_response = cell_data[
            cell_data[parameter_name] == smallest_value
        ]["spike_count"].values[0]

        high_response = cell_data[
            cell_data[parameter_name] == largest_value
        ]["spike_count"].values[0]

        metric_value = metric_func(low_response, high_response)

        rows.append({
            "cell_id": cell_id,
            "Smallest value": smallest_value,
            "Largest value": largest_value,
            "Response at smallest value": low_response,
            "Response at largest value": high_response,
            metric_name: metric_value
        })

    return pd.DataFrame(rows)

## Final function

In [ ]:
def create_general_response_table(
    pipe,
    parameter_name,
    metric_func,
    metric_name,
    cell_types=None,
    minimum_n=3
):
    response_block = pipe.response_block
    stim_block = pipe.stim_block

    exp_date = get_experiment_date(stim_block)

    spike_counts = get_spike_count_table(
        response_block,
        cell_types=cell_types,
        minimum_n=minimum_n
    )

    spike_counts = add_parameter_column(
        spike_counts,
        stim_block,
        parameter_name
    )

    summary = average_response_by_parameter(
        spike_counts,
        parameter_name
    )

    metric_table = calculate_metric_table(
        summary,
        parameter_name,
        metric_func,
        metric_name
    )

    metric_table["Date of experiment"] = exp_date
    metric_table["Parameter name"] = parameter_name

    metric_table = metric_table[
        [
            "cell_id",
            "Date of experiment",
            "Parameter name",
            "Smallest value",
            "Largest value",
            "Response at smallest value",
            "Response at largest value",
            metric_name
        ]
        ]

    return metric_table
    

## Creates OMS final table 

In [ ]:
oms_table = create_general_response_table(
    pipe=pipe,
    parameter_name="spaceConstants",
    metric_func=normalized_difference,
    metric_name="OMS Index"
)

oms_table

/var/folders/gj/d99p19qd5n7bj5cc65_1_v200000gn/T/ipykernel_39104/1696458188.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return (high_response - low_response) / (


,cell_id,Date of experiment,Parameter name,Smallest value,Largest value,Response at smallest value,Response at largest value,OMS Index
0,1,2025-07-15,spaceConstants,50.0,1000.0,906.0,902.0,-0.002212
1,5,2025-07-15,spaceConstants,50.0,1000.0,612.0,354.0,-0.267081
2,8,2025-07-15,spaceConstants,50.0,1000.0,9.0,1.0,-0.800000
3,10,2025-07-15,spaceConstants,50.0,1000.0,7.0,15.0,0.363636
4,11,2025-07-15,spaceConstants,50.0,1000.0,13.0,1.0,-0.857143
...,...,...,...,...,...,...,...,...
781,1429,2025-07-15,spaceConstants,50.0,1000.0,5.0,4.0,-0.111111
782,1431,2025-07-15,spaceConstants,50.0,1000.0,0.0,1.0,1.000000
783,1434,2025-07-15,spaceConstants,50.0,1000.0,0.0,0.0,NaN
784,1439,2025-07-15,spaceConstants,50.0,1000.0,65.0,128.0,0.326425
